In [1]:
from transformers import ViTImageProcessor, ViTModel, ViTConfig
from transformers.utils import cached_file
import torch.nn as nn
import torch 
import glob
import random
from PIL import Image
import json
import torchvision.transforms as T

import sys
sys.path.append("../../../../donut/src/test/")
from common.config import cfg

In [2]:
# load model variables from ckpt file
ckpt_path = "./ViTCheckpoints/20250321-112301/vit-best-epoch=12-val_acc=0.9906.ckpt"
ckpt = torch.load(ckpt_path, map_location="cpu") 

print(ckpt.keys())

# extract state dict content
state_dict = ckpt["state_dict"]

# check the 
for k in state_dict.keys():
    print(k, state_dict[k].shape)

print('***************************')
print(ckpt.get("hyper_parameters", {}))

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'hparams_name', 'hyper_parameters'])
model.embeddings.cls_token torch.Size([1, 1, 768])
model.embeddings.position_embeddings torch.Size([1, 401, 768])
model.embeddings.patch_embeddings.projection.weight torch.Size([768, 3, 16, 16])
model.embeddings.patch_embeddings.projection.bias torch.Size([768])
model.encoder.layer.0.attention.attention.query.weight torch.Size([768, 768])
model.encoder.layer.0.attention.attention.query.bias torch.Size([768])
model.encoder.layer.0.attention.attention.key.weight torch.Size([768, 768])
model.encoder.layer.0.attention.attention.key.bias torch.Size([768])
model.encoder.layer.0.attention.attention.value.weight torch.Size([768, 768])
model.encoder.layer.0.attention.attention.value.bias torch.Size([768])
model.encoder.layer.0.attention.output.dense.weight torch.Size([768, 768])
model.encoder.layer.0.attention.output.dense.bias torch.Size([768])
model.encoder.layer.0.inter

In [3]:
# load variables

local_config_path = cfg.vit_local_config_path

# label encoder
label_encoder = {
    "dot": cfg.dot,
    "scatter": cfg.scatter,
    "horizontal_bar": cfg.horizontal_bar,
    "line": cfg.line,
    "vertical_bar": cfg.vertical_bar,
}

# label decoder
label_decoder = {v: k for k, v in label_encoder.items()}

id2label = {idx: label for idx, label in label_decoder.items()}
label2id = {label: idx for idx, label in label_encoder.items()}

# Dataset parameters
image_cls_height = cfg.vit_image_cls_height
image_cls_width = cfg.vit_image_cls_width

In [4]:
config = ViTConfig.from_pretrained("google/vit-base-patch16-224-in21k",cache_dir=local_config_path)

In [5]:
# add customized paramaters into config file

config.num_labels = len(id2label)
config.id2label = id2label
config.label2id = label2id
config.hidden_dropout_prob = 0.1 
config.image_size = 320
config.patch_size = 16
# 320/16=20,
# input token num: 320*320 / 16*16 = 20*20=400 
# add CLS: 400+1=401
# each token seq len: 16*16*3=768

model_name = "google/vit-base-patch16-224-in21k"
config_path = cached_file(model_name, "config.json")

In [6]:
# load the model
# do not use ViTClassifier
# i only need to inlcude classifier head right here

class SimpleViTClassifier(nn.Module):
    def __init__(self, config, num_classes):
        super().__init__()
        self.vit = ViTModel(config)
        self.classifier = nn.Linear(config.hidden_size, num_classes)

    def forward(self, x):
        outputs = self.vit(x)
        cls_token = outputs.last_hidden_state[:, 0, :]  # [B, hidden]
        return self.classifier(cls_token)

In [7]:
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("model."):
        new_state_dict[k.replace("model.", "vit.")] = v
    elif k.startswith("classifier."):
        new_state_dict[k] = v  

In [8]:
model = SimpleViTClassifier(config=config, num_classes=len(id2label))
model.load_state_dict(new_state_dict)
model.eval()
model.to("mps" if torch.cuda.is_available() else "cpu")
model_device = next(model.parameters()).device 

print(model_device)

cpu


In [9]:
# select images

# list all image paths
image_dir = "../../../data/image_resize/images/"
image_files = glob.glob(image_dir + "*")
print(image_files[:5])

# list all annotation files paths

annotations_dir="../../../data/image_resize/annotations/"
annotations_files=glob.glob(annotations_dir+"*")
print(annotations_files[:5])


# select 20 pics randomly

selected_imgs= random.sample(image_files, 20)
print(selected_imgs)

# extract the related labels
selected_labels=[]
for idx in range(len(selected_imgs)):
    img_path=selected_imgs[idx]
    image_id=img_path.split('/')[-1].split('.')[0]
    anno_path = [file for file in annotations_files if image_id in file]
    with open(anno_path[0],"r") as f:
        annottation=json.load(f)
        selected_labels.append(annottation["chart-type"])



['../../../data/image_resize/images/45df1fe3293b.jpg', '../../../data/image_resize/images/b2ab3b743d4e.jpg', '../../../data/image_resize/images/51d3b1a6baf3.jpg', '../../../data/image_resize/images/a9e9ce9277c1.jpg', '../../../data/image_resize/images/7f1f545fe081.jpg']
['../../../data/image_resize/annotations/e91e28111e86.json', '../../../data/image_resize/annotations/75c0449f6917.json', '../../../data/image_resize/annotations/66dd2a250237.json', '../../../data/image_resize/annotations/58595c30beab.json', '../../../data/image_resize/annotations/497a547454d7.json']
['../../../data/image_resize/images/5a4a42d5fd56.jpg', '../../../data/image_resize/images/1a2a53e18e48.jpg', '../../../data/image_resize/images/3adfb07fa223.jpg', '../../../data/image_resize/images/2cc761314f5d.jpg', '../../../data/image_resize/images/9c21a54c1176.jpg', '../../../data/image_resize/images/48c6a35380ef.jpg', '../../../data/image_resize/images/8c238592ec47.jpg', '../../../data/image_resize/images/d32620031e18.j

In [10]:
transform = T.Compose([
    T.Resize((image_cls_height, image_cls_width)),                    
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])

results=[]
for img in selected_imgs:
    img = Image.open(img).convert("RGB")
    input_tensor = transform(img).unsqueeze(0).to(model_device)

    with torch.no_grad():
        logits = model(input_tensor)
        pred = torch.argmax(logits, dim=1).item()

    results.append(pred)

In [11]:
predicted_labels = [config.id2label[id] for id in results]

In [12]:
predicted_labels

['scatter',
 'vertical_bar',
 'line',
 'line',
 'line',
 'line',
 'line',
 'line',
 'scatter',
 'line',
 'vertical_bar',
 'vertical_bar',
 'line',
 'vertical_bar',
 'line',
 'line',
 'dot',
 'line',
 'vertical_bar',
 'line']

In [13]:
matches = [a == b for a, b in zip(selected_labels, predicted_labels)]

# calculate the accuracy
accuracy = sum(matches) / len(matches)

print(f"accuracy: {accuracy}")

accuracy: 1.0


In [14]:
print(matches)

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
